In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
import torch
x = torch.tensor(3.)
w=torch.tensor(4.,requires_grad=True)
b=torch.tensor(5.,requires_grad=True)

x,w,b

(tensor(3.), tensor(4., requires_grad=True), tensor(5., requires_grad=True))

In [4]:

y=x*w+b
y

tensor(17., grad_fn=<AddBackward0>)

In [5]:
y.backward()


In [6]:
print("dy/dx:",x.grad)
print("dy/dw:",w.grad)
print("dy/db:",b.grad)

dy/dx: None
dy/dw: tensor(3.)
dy/db: tensor(1.)


In [7]:
w.grad

tensor(3.)

Creating a tensor with fixed value 

In [8]:
t1=torch.full((3,2),22)
t1

tensor([[22, 22],
        [22, 22],
        [22, 22]])

In [9]:
t2=torch.sin(t1)
t2

tensor([[-0.0089, -0.0089],
        [-0.0089, -0.0089],
        [-0.0089, -0.0089]])

In [10]:
t3=torch.rand((3,2))

In [11]:
t4=torch.cat((t3,t2))
t4

tensor([[ 0.3987,  0.0778],
        [ 0.0751,  0.4996],
        [ 0.1421,  0.1525],
        [-0.0089, -0.0089],
        [-0.0089, -0.0089],
        [-0.0089, -0.0089]])

In [12]:
t5=t4.reshape(3,2,2)
t5

tensor([[[ 0.3987,  0.0778],
         [ 0.0751,  0.4996]],

        [[ 0.1421,  0.1525],
         [-0.0089, -0.0089]],

        [[-0.0089, -0.0089],
         [-0.0089, -0.0089]]])

In [13]:
x=np.array([[1,2],[3,4]])
x

array([[1, 2],
       [3, 4]])

In [14]:
y=torch.from_numpy(x)
y

tensor([[1, 2],
        [3, 4]])

In [15]:
x=y.numpy()
x

array([[1, 2],
       [3, 4]])

Linear regression from scratch

In [16]:
#Input (temp, rainfall, humidity)
ip = np.array([
    [73, 67, 43],
    [91, 88, 64],
    [87, 134, 58],
    [102, 43, 37],
    [69, 96, 70]
], dtype='float32')

# Targets (apples, oranges)
targets = np.array([
    [56, 70],
    [81, 101],
    [119, 133],
    [22, 37],
    [103, 119]
], dtype='float32')

In [17]:
# Already tensors
print(ip, "\n")
print(targets)


[[ 73.  67.  43.]
 [ 91.  88.  64.]
 [ 87. 134.  58.]
 [102.  43.  37.]
 [ 69.  96.  70.]] 

[[ 56.  70.]
 [ 81. 101.]
 [119. 133.]
 [ 22.  37.]
 [103. 119.]]


In [26]:
import torch

# Convert input and target to tensors
ip=torch.from_numpy(ip)
target = torch.from_numpy(targets)

print(ip, "\n")
print(target)


tensor([[ 73.,  67.,  43.],
        [ 91.,  88.,  64.],
        [ 87., 134.,  58.],
        [102.,  43.,  37.],
        [ 69.,  96.,  70.]]) 

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])


In [27]:
w=torch.randn(2,3,requires_grad=True)
b=torch.randn(2,requires_grad=True)
w,b

(tensor([[-0.6140, -1.0098,  0.8669],
         [-2.0085,  1.0876,  0.7621]], requires_grad=True),
 tensor([1.1241, 2.4365], requires_grad=True))

In [28]:
def model(x):
    return x @ w.t() +b


In [29]:
preds=model(ip)
preds

tensor([[ -74.0826,  -38.5453],
        [ -88.1371,  -35.8553],
        [-137.3350,   17.6353],
        [ -72.8539, -127.4654],
        [ -77.5064,   21.6042]], grad_fn=<AddBackward0>)

In [30]:
def MSE(true,target):
    diff=true-target
    return torch.sum(diff*diff)/diff.numel()


In [31]:
loss=MSE(target,preds)
loss

tensor(23317.1543, grad_fn=<DivBackward0>)

In [32]:
#Compute gradiant
loss.backward()

In [33]:
w.grad,b.grad

(tensor([[-13863.7363, -15871.1631,  -9486.1592],
         [-10782.0303, -10239.3369,  -6604.0537]]),
 tensor([-166.1830, -124.5253]))

In [34]:
#REset grad
w.grad.zero_()
b.grad.zero_()

w.grad,b.grad

(tensor([[0., 0., 0.],
         [0., 0., 0.]]),
 tensor([0., 0.]))

In [35]:
pred=model(ip)
pred

tensor([[ -74.0826,  -38.5453],
        [ -88.1371,  -35.8553],
        [-137.3350,   17.6353],
        [ -72.8539, -127.4654],
        [ -77.5064,   21.6042]], grad_fn=<AddBackward0>)

In [36]:
#Loss
loss=MSE(target,pred)
loss

tensor(23317.1543, grad_fn=<DivBackward0>)

In [37]:
loss.backward()

In [38]:
w.grad,b.grad

(tensor([[-13863.7363, -15871.1631,  -9486.1592],
         [-10782.0303, -10239.3369,  -6604.0537]]),
 tensor([-166.1830, -124.5253]))

In [39]:
with torch.no_grad():
    w-=w.grad * 1e-5
    b-=b.grad * 1e-5
    w.grad.zero_()
    b.grad.zero_()

In [40]:
w,b

(tensor([[-0.4754, -0.8511,  0.9617],
         [-1.9007,  1.1900,  0.8281]], requires_grad=True),
 tensor([1.1257, 2.4377], requires_grad=True))

In [41]:
ip.dtype

torch.float32

In [42]:
for i in range(400):
    preds=model(ip)
    loss=MSE(target,preds)
    loss.backward()

    with torch.no_grad():
        w-=w.grad * 1e-5#Learning rate
        b-=b.grad * 1e-5
        w.grad.zero_()
        b.grad.zero_()
    print(f"Epochs({i}/{100}) & Loss {loss}")

preds=model(ip)
loss=MSE(target,preds)
loss

Epochs(0/100) & Loss 16043.009765625
Epochs(1/100) & Loss 11137.0283203125
Epochs(2/100) & Loss 7826.9775390625
Epochs(3/100) & Loss 5592.4462890625
Epochs(4/100) & Loss 4082.74462890625
Epochs(5/100) & Loss 3061.54248046875
Epochs(6/100) & Loss 2369.582763671875
Epochs(7/100) & Loss 1899.541015625
Epochs(8/100) & Loss 1579.094482421875
Epochs(9/100) & Loss 1359.5035400390625
Epochs(10/100) & Loss 1207.9241943359375
Epochs(11/100) & Loss 1102.221923828125
Epochs(12/100) & Loss 1027.4801025390625
Epochs(13/100) & Loss 973.6461791992188
Epochs(14/100) & Loss 933.9454956054688
Epochs(15/100) & Loss 903.8118896484375
Epochs(16/100) & Loss 880.1680908203125
Epochs(17/100) & Loss 860.9392700195312
Epochs(18/100) & Loss 844.7271728515625
Epochs(19/100) & Loss 830.5885620117188
Epochs(20/100) & Loss 817.8878173828125
Epochs(21/100) & Loss 806.1961059570312
Epochs(22/100) & Loss 795.2235107421875
Epochs(23/100) & Loss 784.7740478515625
Epochs(24/100) & Loss 774.7159423828125
Epochs(25/100) & Lo

tensor(24.3107, grad_fn=<DivBackward0>)

tensor(5.3017, grad_fn=<DivBackward0>)

In [43]:
from math import sqrt
sqrt(loss)

C:\Users\kanan\AppData\Local\Temp\ipykernel_13200\3167310485.py:2: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  sqrt(loss)


4.930591627296466

In [44]:
target

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])

In [45]:
preds

tensor([[ 57.5065,  69.7450],
        [ 86.7130, 100.3880],
        [107.8994, 134.5147],
        [ 22.4012,  31.3652],
        [109.2116, 122.4273]], grad_fn=<AddBackward0>)

## Neural Network using python

In [46]:
import torch
print(torch.cuda.is_available())  # will print False


False


In [47]:
import torch
from torch import nn
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor,Lambda,Compose
import matplotlib.pyplot as plt
import seaborn as sns

In [48]:
# Download training data from open datasets
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [49]:
type(training_data)

torchvision.datasets.mnist.FashionMNIST

In [50]:
batch_size=64
train_dataloader=DataLoader(training_data,batch_size=batch_size)
test_dataloader=DataLoader(test_data,batch_size=batch_size)

for X,y in test_dataloader:
    print("Shape of X [N, C, H, W]: ",X.shape)
    print("Shape of y :",y.shape,y.dtype)

    break

Shape of X [N, C, H, W]:  torch.Size([64, 1, 28, 28])
Shape of y : torch.Size([64]) torch.int64


In [51]:
# Get CPU device for training
device = "cpu"
print(f"Using {device} device")


Using cpu device


In [52]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork,self).__init__()
        self.flatten=nn.Flatten()
        self.linear_relu_stack=nn.Sequential(
            nn.Linear(28*28,512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )

    def forward(self,x):
        x=self.flatten(x)
        logits=self.linear_relu_stack(x)
        return logits


model=NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [53]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(),lr=1e-3)
loss_fn
optimizer

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

In [54]:
def train(dataloader,model,loss_fn,optimizer):
    size=len(dataloader.dataset)
    model.train()
    for batch, (X,y) in enumerate(dataloader):
        X,y=X.to(device),y.to(device)

        #Compute prediction error

        pred=model(X)
        loss=loss_fn(pred,y)

        #BackPropogation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss,current=loss.item() , batch * len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]" )


In [55]:
def test(dataloader,model,loss_fn):
    size=len(dataloader.dataset)
    num_batches=len(dataloader)
    model.eval()
    test_loss,correct=0 , 0
    with torch.no_grad():
        for X,y in dataloader:
            X,y=X.to(device),y.to(device)
            pred=model(X)
            test_loss+= loss_fn(pred,y).item()
            correct+=(pred.argmax(1)==y).type(torch.float).sum().item()
    
    test_loss /= num_batches
    correct /= size
    print(
    f"Test Error:\n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f}\n")    

In [56]:
epochs = 5

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)

print("Done!")


Epoch 1
-------------------------------
loss: 2.292702 [    0/60000]
loss: 2.290669 [ 6400/60000]
loss: 2.271099 [12800/60000]
loss: 2.266774 [19200/60000]
loss: 2.256161 [25600/60000]
loss: 2.217577 [32000/60000]
loss: 2.230598 [38400/60000]
loss: 2.193935 [44800/60000]
loss: 2.178651 [51200/60000]
loss: 2.160723 [57600/60000]
Test Error:
 Accuracy: 39.0%, Avg loss: 2.159924

Epoch 2
-------------------------------
loss: 2.162812 [    0/60000]
loss: 2.160104 [ 6400/60000]
loss: 2.104735 [12800/60000]
loss: 2.119852 [19200/60000]
loss: 2.074798 [25600/60000]
loss: 2.010092 [32000/60000]
loss: 2.036954 [38400/60000]
loss: 1.958434 [44800/60000]
loss: 1.952492 [51200/60000]
loss: 1.892139 [57600/60000]
Test Error:
 Accuracy: 59.9%, Avg loss: 1.896899

Epoch 3
-------------------------------
loss: 1.923246 [    0/60000]
loss: 1.901579 [ 6400/60000]
loss: 1.784791 [12800/60000]
loss: 1.819876 [19200/60000]
loss: 1.718789 [25600/60000]
loss: 1.665285 [32000/60000]
loss: 1.679070 [38400/6000

In [57]:
import pickle

In [58]:
torch.save(model.state_dict(),"model.pth")




In [59]:
model=NeuralNetwork()
model.load_state_dict(torch.load("model.pth"))

<All keys matched successfully>

In [61]:
## Prediction
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    pred = model(x)
predicted, actual = classes[pred[0].argmax(0)], classes[y]
print(f"Predicted: {predicted}, Actual: {actual}")


Predicted: Ankle boot, Actual: Ankle boot
